# Контракт данных и автоматический шлюз качества

Рабочая задача: до запуска аналитики проверить, что входные таблицы соответствуют контракту.  
В notebook используются две стадии:

1. **Raw gate** — проверка исходных файлов. Здесь ожидаются нарушения, специально заложенные в учебные данные.
2. **Clean gate** — проверка результатов очистки. Критические проверки должны пройти.

Результат работы:

- таблица проверок с уровнями `PASS`, `WARNING`, `FAIL`;
- решение `CONTINUE` или `STOP`;
- файл `outputs/data_validation_gate.csv`;
- краткое объяснение, какие нарушения блокируют дальнейший анализ.

> **Место в производственном маршруте:** 1 из 9  
> **Ориентир очного занятия:** 20–35 минут и возврат после notebook 01  
> **Режим:** Основной практический маршрут  
> **Выход этапа:** Raw gate → очистка → clean gate

Студенческая версия содержит задания и контрольные точки без полного решения.

## Порядок выполнения

1. Выполните raw gate в этом notebook.
2. Перейдите в `01_data_quality_eda` и создайте чистые витрины.
3. Вернитесь сюда и выполните clean gate.

Raw gate должен показать, почему запуск блокируется; clean gate — подтвердить готовность данных к моделированию.

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data" / "raw" / "tickets.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Не найдена папка проекта. Убедитесь, что notebook находится внутри распакованного комплекта."
    )


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
CHART_DIR = OUTPUT_DIR / "charts"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [OUTPUT_DIR, CHART_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version.split()[0])
print("Корень проекта:", PROJECT_ROOT)

## Задание 1. Опишите контракт

Для каждой таблицы укажите:

- гранулярность;
- первичный ключ;
- внешние ключи;
- обязательные поля;
- критические правила;
- предупреждающие правила.

Заполните таблицу в markdown-ячейке ниже.

| Таблица | Гранулярность | Первичный ключ | Критическое правило |
|---|---|---|---|
| `tickets` | ... | ... | ... |
| `ticket_events` | ... | ... | ... |
| `team_capacity_daily` | ... | ... | ... |

In [ ]:
# Загружаем исходные файлы. Дополните список по необходимости.
raw_tickets = pd.read_csv(RAW_DIR / "tickets.csv", low_memory=False)
raw_events = pd.read_csv(RAW_DIR / "ticket_events.csv", low_memory=False)
raw_capacity = pd.read_csv(RAW_DIR / "team_capacity_daily.csv", low_memory=False)
teams = pd.read_excel(RAW_DIR / "teams.xlsx")

print("tickets:", raw_tickets.shape)
print("events:", raw_events.shape)
print("capacity:", raw_capacity.shape)

## Задание 2. Реализуйте проверки raw gate

In [ ]:
checks = []

# Пример готовой проверки.
checks.append({
    "dataset": "tickets",
    "check_name": "file_not_empty",
    "severity": "critical",
    "status": "PASS" if len(raw_tickets) > 0 else "FAIL",
    "observed": len(raw_tickets),
    "expected": "> 0 rows",
})

# TODO 1: уникальность ticket_id.
# TODO 2: отсутствие пропусков created_at_utc.
# TODO 3: валидность team_id относительно teams.xlsx.
# TODO 4: уникальность event_id.
# TODO 5: отсутствие orphan events.
# TODO 6: уникальность ключа date + team_id.

raw_report = pd.DataFrame(checks)
display(raw_report)

## Задание 3. Сформируйте решение шлюза

Правило:

- если есть хотя бы один `FAIL` с `severity == "critical"`, решение равно `STOP`;
- иначе решение равно `CONTINUE`.

In [ ]:
# TODO: рассчитайте gate_decision.
gate_decision = "TODO"
print("RAW GATE:", gate_decision)

## Задание 4. Повторите проверки после очистки

In [ ]:
# Сначала выполните notebook 01_data_quality_eda_student.ipynb.
clean_path = OUTPUT_DIR / "tickets_clean.csv"
if clean_path.exists():
    clean_tickets = pd.read_csv(clean_path, parse_dates=["created_date", "created_at_utc"])
    print(clean_tickets.shape)
else:
    print("Не найден tickets_clean.csv. Сначала выполните notebook 01.")

# TODO: проверьте число строк, уникальность ticket_id, внешние ключи и диапазоны значений.

## Что сохранить

Сохраните таблицу с полями:

```text
stage, dataset, check_name, severity, status, observed, expected, details
```

Имя файла: `outputs/data_validation_gate.csv`.